In [ ]:
# Install Playwright (uncomment and run if not installed)
# !python -m pip install playwright
# !python -m playwright install chromium

OUTPUT_PLAYWRIGHT_TEST = '/workspace/cars24_playwright_test.csv'  # target path; may not be writable
FALLBACK_OUTPUT = '~/cars24_playwright_test.csv'
HTML_SNAPSHOT_PATH = '/workspace/cars24_hyderabad_snapshot.html'
FALLBACK_HTML_SNAPSHOT = '~/cars24_hyderabad_snapshot.html'
CITY = 'Hyderabad'
CITY_URL = 'https://www.cars24.com/buy-used-cars-hyderabad/'
MAX_LISTINGS = 20
REQUEST_DELAY = 2

In [ ]:
from playwright.sync_api import sync_playwright, TimeoutError as PlaywrightTimeoutError
from bs4 import BeautifulSoup
import pandas as pd
import time, re, os, datetime

def detect_blocking(page_content):
    s = page_content.lower()
    keys = ['captcha','access denied','are you human','verify','unusual traffic','bot verification','blocked']
    for k in keys:
        if k in s:
            return True, k
    return False, None

# Try multiple selector strategies to locate listing cards on the rendered page
CARD_SELECTORS = [
    'a[href*=
]',
    'a[href*=
]',
    'div[data-listing-id] a',
    'article a[href*=
]',
    'a[class*=ListingCard], a[class*=listing]'
: 
,
: {
: 

: [
,
,
,
,
,
,
,
,
,
,
,

In [1]:
import pandas as pd

df = pd.read_csv("cars24_playwright_test.csv")

print("Total records:", len(df))
print("Columns:", df.columns.tolist())

print("\nFuel types:")
print(df["fuel_type"].value_counts(dropna=False))

print("\nTransmissions:")
print(df["transmission"].value_counts(dropna=False))

print("\nUnique URLs:", df["listing_url"].nunique())
print("Duplicate URLs:", df["listing_url"].duplicated().sum())

display(df.head(20))


Total records: 20
Columns: ['listing_url', 'listing_id', 'car_name', 'price', 'manufacturing_year', 'km_driven', 'fuel_type', 'transmission', 'brand', 'model', 'variant', 'location', 'city', 'state']

Fuel types:
fuel_type
Petrol    20
Name: count, dtype: int64

Transmissions:
transmission
Automatic    20
Name: count, dtype: int64

Unique URLs: 20
Duplicate URLs: 0


,listing_url,listing_id,car_name,price,manufacturing_year,km_driven,fuel_type,transmission,brand,model,variant,location,city,state
0,https://www.cars24.com/buy-used-nissan-kicks-c...,NaN,5 Second Hand Nissan Kicks in Hyderabad,"EMI ₹11,925/m*",2019,"1,18,342",Petrol,Automatic,5,5 Second,Hand Nissan,Hyderabad,Hyderabad,NaN
1,https://www.cars24.com/buy-used-sunroof-cars-h...,NaN,535 Used Sunroof cars in Hyderabad,535 Second Hand Sunroof Cars in Hyderabad | Us...,2022,"54,649",Petrol,Automatic,535,535 Used,Sunroof cars,Hyderabad,Hyderabad,NaN
2,https://www.cars24.com/buy-used-honda-amaze-ca...,NaN,38 Second Hand Honda Amaze in Hyderabad,₹2.50 lakh - ₹3.60 lakh,2013,46,Petrol,Automatic,38,38 Second,Hand Honda,Hyderabad,Hyderabad,NaN
3,https://www.cars24.com/buy-used-mahindra-xuv50...,NaN,32 Second Hand Mahindra XUV500 in Hyderabad,"EMI ₹15,952/m*",2011,1,Petrol,Automatic,32,32 Second,Hand Mahindra,Hyderabad,Hyderabad,NaN
4,https://www.cars24.com/buy-used-renault-kwid-2...,NaN,2020Renault KwidCLIMBER 1.0 (O),"₹6,371/mo",2020,"33,425",Petrol,Automatic,2020Renault,2020Renault KwidCLIMBER,1.0 (O),"Bachupally, Hyderabad",Hyderabad,NaN
5,https://www.cars24.com/buy-used-maruti-s-press...,NaN,7 Second Hand Maruti S PRESSO in Hyderabad,₹3.70 lakh - ₹4.00 lakh,2020,21,Petrol,Automatic,7,7 Second,Hand Maruti,Hyderabad,Hyderabad,NaN
6,https://www.cars24.com/buy-used-audi-a4-cars-h...,NaN,4 Second Hand Audi A4 in Hyderabad,₹6.70 lakh - ₹14.90 lakh,2015,"1,06,735",Petrol,Automatic,4,4 Second,Hand Audi,Hyderabad,Hyderabad,NaN
7,https://www.cars24.com/buy-used-bmw-5-series-c...,NaN,2 Second Hand BMW 5 Series in Hyderabad,₹10.50 lakh - ₹29.00 lakh,2012,"75,696",Petrol,Automatic,2,2 Second,Hand BMW,Hyderabad,Hyderabad,NaN
8,https://www.cars24.com/buy-used-skoda-karoq-ca...,NaN,2 Second Hand Skoda Karoq in Hyderabad,"EMI ₹22,784/m*",2020,"1,05,516",Petrol,Automatic,2,2 Second,Hand Skoda,Hyderabad,Hyderabad,NaN
9,https://www.cars24.com/buy-used-ford-figo-cars...,NaN,21 Second Hand Ford Figo in Hyderabad,₹1.00 lakh - ₹1.36 lakh,2011,20,Petrol,Automatic,21,21 Second,Hand Ford,Hyderabad,Hyderabad,NaN


In [2]:
print("Sample scraped data:")

display(
    df[
        [
            "listing_url",
            "car_name",
            "price",
            "manufacturing_year",
            "km_driven",
            "fuel_type",
            "transmission",
            "brand",
            "model",
            "variant",
            "location",
            "city",
            "state"
        ]
    ].head(20)
)

Sample scraped data:


,listing_url,car_name,price,manufacturing_year,km_driven,fuel_type,transmission,brand,model,variant,location,city,state
0,https://www.cars24.com/buy-used-nissan-kicks-c...,5 Second Hand Nissan Kicks in Hyderabad,"EMI ₹11,925/m*",2019,"1,18,342",Petrol,Automatic,5,5 Second,Hand Nissan,Hyderabad,Hyderabad,NaN
1,https://www.cars24.com/buy-used-sunroof-cars-h...,535 Used Sunroof cars in Hyderabad,535 Second Hand Sunroof Cars in Hyderabad | Us...,2022,"54,649",Petrol,Automatic,535,535 Used,Sunroof cars,Hyderabad,Hyderabad,NaN
2,https://www.cars24.com/buy-used-honda-amaze-ca...,38 Second Hand Honda Amaze in Hyderabad,₹2.50 lakh - ₹3.60 lakh,2013,46,Petrol,Automatic,38,38 Second,Hand Honda,Hyderabad,Hyderabad,NaN
3,https://www.cars24.com/buy-used-mahindra-xuv50...,32 Second Hand Mahindra XUV500 in Hyderabad,"EMI ₹15,952/m*",2011,1,Petrol,Automatic,32,32 Second,Hand Mahindra,Hyderabad,Hyderabad,NaN
4,https://www.cars24.com/buy-used-renault-kwid-2...,2020Renault KwidCLIMBER 1.0 (O),"₹6,371/mo",2020,"33,425",Petrol,Automatic,2020Renault,2020Renault KwidCLIMBER,1.0 (O),"Bachupally, Hyderabad",Hyderabad,NaN
5,https://www.cars24.com/buy-used-maruti-s-press...,7 Second Hand Maruti S PRESSO in Hyderabad,₹3.70 lakh - ₹4.00 lakh,2020,21,Petrol,Automatic,7,7 Second,Hand Maruti,Hyderabad,Hyderabad,NaN
6,https://www.cars24.com/buy-used-audi-a4-cars-h...,4 Second Hand Audi A4 in Hyderabad,₹6.70 lakh - ₹14.90 lakh,2015,"1,06,735",Petrol,Automatic,4,4 Second,Hand Audi,Hyderabad,Hyderabad,NaN
7,https://www.cars24.com/buy-used-bmw-5-series-c...,2 Second Hand BMW 5 Series in Hyderabad,₹10.50 lakh - ₹29.00 lakh,2012,"75,696",Petrol,Automatic,2,2 Second,Hand BMW,Hyderabad,Hyderabad,NaN
8,https://www.cars24.com/buy-used-skoda-karoq-ca...,2 Second Hand Skoda Karoq in Hyderabad,"EMI ₹22,784/m*",2020,"1,05,516",Petrol,Automatic,2,2 Second,Hand Skoda,Hyderabad,Hyderabad,NaN
9,https://www.cars24.com/buy-used-ford-figo-cars...,21 Second Hand Ford Figo in Hyderabad,₹1.00 lakh - ₹1.36 lakh,2011,20,Petrol,Automatic,21,21 Second,Hand Ford,Hyderabad,Hyderabad,NaN


In [3]:
# Check which URLs were scraped
import pandas as pd

df = pd.read_csv("cars24_playwright_test.csv")

print("Total records:", len(df))

print("\nScraped URLs:")
for i, url in enumerate(df["listing_url"].head(20), 1):
    print(i, url)

print("\nURL patterns:")
print(
    df["listing_url"]
    .str.extract(r'cars24\.com/(.*?)/?$')[0]
    .head(20)
)

Total records: 20

Scraped URLs:
1 https://www.cars24.com/buy-used-nissan-kicks-cars-hyderabad/
2 https://www.cars24.com/buy-used-sunroof-cars-hyderabad/
3 https://www.cars24.com/buy-used-honda-amaze-cars-hyderabad/
4 https://www.cars24.com/buy-used-mahindra-xuv500-cars-hyderabad/
5 https://www.cars24.com/buy-used-renault-kwid-2020-cars-hyderabad-10457877130/
6 https://www.cars24.com/buy-used-maruti-s-presso-cars-hyderabad/
7 https://www.cars24.com/buy-used-audi-a4-cars-hyderabad/
8 https://www.cars24.com/buy-used-bmw-5-series-cars-hyderabad/
9 https://www.cars24.com/buy-used-skoda-karoq-cars-hyderabad/
10 https://www.cars24.com/buy-used-ford-figo-cars-hyderabad/
11 https://www.cars24.com/buy-used-manual-cars-hyderabad/
12 https://www.cars24.com/buy-used-mahindra-cars-hyderabad/
13 https://www.cars24.com/buy-used-jaguar-cars-hyderabad/
14 https://www.cars24.com/buy-used-volkswagen-taigun-cars-hyderabad/
15 https://www.cars24.com/buy-used-honda-wr-v-cars-hyderabad/
16 https://www.cars24

In [4]:
import re
import pandas as pd

df = pd.read_csv("cars24_playwright_test.csv")

# Keep only URLs that look like individual Cars24 listings
# Individual listing URLs contain a numeric listing ID at the end.
pattern = r"cars24\.com/.+-hyderabad-\d+/?$"

df_individual = df[
    df["listing_url"].astype(str).str.contains(
        pattern,
        regex=True,
        case=False,
        na=False
    )
].copy()

print("Total scraped URLs:", len(df))
print("Individual listing URLs:", len(df_individual))
print("Category/filter URLs removed:", len(df) - len(df_individual))

display(df_individual)

Total scraped URLs: 20
Individual listing URLs: 1
Category/filter URLs removed: 19


,listing_url,listing_id,car_name,price,manufacturing_year,km_driven,fuel_type,transmission,brand,model,variant,location,city,state
4,https://www.cars24.com/buy-used-renault-kwid-2...,NaN,2020Renault KwidCLIMBER 1.0 (O),"₹6,371/mo",2020,"33,425",Petrol,Automatic,2020Renault,2020Renault KwidCLIMBER,1.0 (O),"Bachupally, Hyderabad",Hyderabad,NaN


In [5]:
import re
from playwright.async_api import async_playwright

url = "https://www.cars24.com/buy-used-renault-kwid-2020-cars-hyderabad-10457877130/"

async def inspect_listing():
    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=True)

        page = await browser.new_page(
            viewport={"width": 1440, "height": 900}
        )

        print("Opening listing...")

        await page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=60000
        )

        await page.wait_for_timeout(5000)

        print("Title:", await page.title())

        text = await page.locator("body").inner_text()

        print("\n========== LISTING TEXT ==========\n")
        print(text[:10000])

        await browser.close()


await inspect_listing()

Opening listing...
Title: Used 2020 Renault Kwid CLIMBER 1.0 (O) Manual in Hyderabad | 33,425 Kms - Cars24

========== LISTING TEXT ==========

Hyderabad

Buy used car

Sell car

Car finance

New cars

Car services

Call us

Hello, Sign in

Account

Overview

Car inspection report

Features and specs

 Previous

Reasons to buy this car

Smart connectivity
Fuel-efficient car
GPS connectivity

Top features of this car

ABS - Anti-lock Braking System
Air Conditioner
Airbags
Bluetooth Compatibility

Why choose Cars24?

30-day easy returns
Lifetime Warranty Upgrade
Assured buyback
 Next

Exterior

Interior

Features

Highlights

Tyres

Great things about this car

Smart connectivity

with Apple CarPlay and Android Auto.

Fuel-efficient car

Drive more, spend less.

GPS connectivity

Precise navigation with in-built GPS.

Multitasking infotainment

8.0-inch screen for a rich experience.

Essential features

AC, power windows, music system.

184 mm ground clearance

High clearance for bumpy r

In [6]:
from playwright.async_api import async_playwright

url = "https://www.cars24.com/buy-used-renault-kwid-2020-cars-hyderabad-10457877130/"

async def inspect_structured_data():

    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=True)

        page = await browser.new_page(
            viewport={"width": 1440, "height": 900}
        )

        await page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=60000
        )

        await page.wait_for_timeout(5000)

        # Check JSON-LD
        json_ld = await page.locator(
            'script[type="application/ld+json"]'
        ).all_text_contents()

        print("========== JSON-LD DATA ==========")
        print("Number of JSON-LD scripts:", len(json_ld))

        for i, data in enumerate(json_ld):
            print(f"\n--- JSON-LD {i+1} ---")
            print(data[:5000])

        # Check Next.js / page scripts containing vehicle information
        scripts = await page.locator("script").all_text_contents()

        print("\n========== VEHICLE DATA IN SCRIPTS ==========")

        found = 0

        for i, script in enumerate(scripts):

            if any(word.lower() in script.lower()
                   for word in [
                       "Renault",
                       "Kwid",
                       "CLIMBER",
                       "369940",
                       "370000",
                       "33425"
                   ]):

                print(f"\n--- Matching script {i} ---")
                print(script[:8000])

                found += 1

                if found >= 5:
                    break

        print("\nMatching scripts found:", found)

        await browser.close()


await inspect_structured_data()

========== JSON-LD DATA ==========
Number of JSON-LD scripts: 3

--- JSON-LD 1 ---
[{"@context":"https://schema.org/","@id":"https://www.cars24.com/","url":"https://www.cars24.com/","name":"Cars24","@type":"WebPage"},{"@context":"https://schema.org/","@id":"https://www.cars24.com/","url":"https://www.cars24.com/","name":"Cars24","@type":"Website"}]

--- JSON-LD 2 ---
{"@context":"http://schema.org","@type":"BreadcrumbList","itemListElement":[{"@type":"ListItem","position":1,"item":{"@id":"https://www.cars24.com/","name":"Home"}},{"@type":"ListItem","position":2,"item":{"@id":"https://www.cars24.com/buy-used-cars/","name":"Used Cars"}},{"@type":"ListItem","position":3,"item":{"@id":"https://www.cars24.com/buy-used-cars-hyderabad/","name":"Used Cars in Hyderabad"}},{"@type":"ListItem","position":4,"item":{"@id":"https://www.cars24.com/buy-used-renault-cars-hyderabad/","name":"Used Renault Cars in Hyderabad"}},{"@type":"ListItem","position":5,"item":{"@id":"https://www.cars24.com/buy-used

In [7]:
    import json
import pandas as pd
from playwright.async_api import async_playwright

url = "https://www.cars24.com/buy-used-renault-kwid-2020-cars-hyderabad-10457877130/"

async def extract_car(url):

    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=True)

        page = await browser.new_page(
            viewport={"width": 1440, "height": 900}
        )

        await page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=60000
        )

        await page.wait_for_timeout(3000)

        json_ld_scripts = await page.locator(
            'script[type="application/ld+json"]'
        ).all_text_contents()

        car_data = None

        for script in json_ld_scripts:

            try:
                data = json.loads(script)

                # JSON-LD can be a list or dictionary
                items = data if isinstance(data, list) else [data]

                for item in items:

                    if (
                        isinstance(item, dict)
                        and item.get("@type") == "Car"
                    ):
                        car_data = item
                        break

            except Exception:
                continue

            if car_data:
                break

        await browser.close()

        if not car_data:
            return None

        # Extract values
        brand = car_data.get("brand", {})
        brand = (
            brand.get("name")
            if isinstance(brand, dict)
            else brand
        )

        offer = car_data.get("offers", {})

        mileage = car_data.get("mileageFromOdometer", {})

        record = {
            "listing_url": url,
            "car_name": car_data.get("name"),
            "brand": brand,
            "model": car_data.get("model"),
            "manufacturing_year": car_data.get("vehicleModelDate"),
            "body_type": car_data.get("bodyType"),
            "fuel_type": car_data.get("fuelType"),
            "transmission": car_data.get("vehicleTransmission"),
            "price": offer.get("price"),
            "price_currency": offer.get("priceCurrency"),
            "km_driven": mileage.get("value"),
        }

        return record


record = await extract_car(url)

df_test = pd.DataFrame([record])

display(df_test)

IndentationError: unexpected indent (3808823141.py, line 1)

In [8]:
import json
import pandas as pd
from playwright.async_api import async_playwright

url = "https://www.cars24.com/buy-used-renault-kwid-2020-cars-hyderabad-10457877130/"

async def extract_car(url):

    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=True)

        page = await browser.new_page(
            viewport={"width": 1440, "height": 900}
        )

        await page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=60000
        )

        await page.wait_for_timeout(3000)

        json_ld_scripts = await page.locator(
            'script[type="application/ld+json"]'
        ).all_text_contents()

        car_data = None

        for script in json_ld_scripts:

            try:
                data = json.loads(script)

                # JSON-LD can be a list or dictionary
                items = data if isinstance(data, list) else [data]

                for item in items:

                    if (
                        isinstance(item, dict)
                        and item.get("@type") == "Car"
                    ):
                        car_data = item
                        break

            except Exception:
                continue

            if car_data:
                break

        await browser.close()

        if not car_data:
            return None

        # Extract values
        brand = car_data.get("brand", {})
        brand = (
            brand.get("name")
            if isinstance(brand, dict)
            else brand
        )

        offer = car_data.get("offers", {})

        mileage = car_data.get("mileageFromOdometer", {})

        record = {
            "listing_url": url,
            "car_name": car_data.get("name"),
            "brand": brand,
            "model": car_data.get("model"),
            "manufacturing_year": car_data.get("vehicleModelDate"),
            "body_type": car_data.get("bodyType"),
            "fuel_type": car_data.get("fuelType"),
            "transmission": car_data.get("vehicleTransmission"),
            "price": offer.get("price"),
            "price_currency": offer.get("priceCurrency"),
            "km_driven": mileage.get("value"),
        }

        return record


record = await extract_car(url)

df_test = pd.DataFrame([record])

display(df_test)

,listing_url,car_name,brand,model,manufacturing_year,body_type,fuel_type,transmission,price,price_currency,km_driven
0,https://www.cars24.com/buy-used-renault-kwid-2...,2020 Renault Kwid CLIMBER 1.0 (O),Renault,Kwid,2020,Hatchback,Petrol,Manual,369940,INR,33425


In [9]:
import re
from playwright.async_api import async_playwright

CITY_URL = "https://www.cars24.com/buy-used-cars-hyderabad/"

async def collect_listing_urls():

    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=True)

        page = await browser.new_page(
            viewport={"width": 1440, "height": 900}
        )

        print("Opening Cars24 Hyderabad...")

        await page.goto(
            CITY_URL,
            wait_until="domcontentloaded",
            timeout=60000
        )

        await page.wait_for_timeout(5000)

        # Scroll to load more cars
        for i in range(8):

            await page.mouse.wheel(0, 4000)

            await page.wait_for_timeout(2000)

            print(f"Scroll {i + 1}/8 completed")

        # Get all links
        links = await page.locator("a[href]").evaluate_all(
            """els => els.map(a => ({
                href: a.href,
                text: (a.innerText || '').trim()
            }))"""
        )

        listing_urls = []

        for item in links:

            href = item["href"]

            # Remove query parameters
            href = href.split("?")[0].rstrip("/")

            # Actual listing pages contain numeric listing ID
            if re.search(r"-\d{8,}$", href):

                listing_urls.append(href + "/")

        # Remove duplicates
        listing_urls = list(dict.fromkeys(listing_urls))

        print("\n====================================")
        print("TOTAL LINKS FOUND:", len(links))
        print("INDIVIDUAL LISTINGS:", len(listing_urls))
        print("====================================")

        print("\nFirst 20 listing URLs:\n")

        for i, url in enumerate(listing_urls[:20], 1):

            print(i, url)

        await browser.close()

        return listing_urls


listing_urls = await collect_listing_urls()

Opening Cars24 Hyderabad...
Scroll 1/8 completed
Scroll 2/8 completed
Scroll 3/8 completed
Scroll 4/8 completed
Scroll 5/8 completed
Scroll 6/8 completed
Scroll 7/8 completed
Scroll 8/8 completed

TOTAL LINKS FOUND: 499
INDIVIDUAL LISTINGS: 20

First 20 listing URLs:

1 https://www.cars24.com/buy-used-maruti-swift-2024-cars-hyderabad-10440373131/
2 https://www.cars24.com/buy-used-renault-kwid-2017-cars-hyderabad-10473477137/
3 https://www.cars24.com/buy-used-maruti-swift-2018-cars-hyderabad-10432970186/
4 https://www.cars24.com/buy-used-tata-nexon-2022-cars-hyderabad-10425496120/
5 https://www.cars24.com/buy-used-renault-kwid-2020-cars-hyderabad-10457877130/
6 https://www.cars24.com/buy-used-tata-nexon-2020-cars-hyderabad-10420372181/
7 https://www.cars24.com/buy-used-volkswagen-polo-2017-cars-hyderabad-10420673116/
8 https://www.cars24.com/buy-used-hyundai-elite-i20-2017-cars-hyderabad-10488773188/
9 https://www.cars24.com/buy-used-ford-ecosport-2019-cars-hyderabad-10481879115/
10 htt

In [10]:
import asyncio
import json
import pandas as pd
from playwright.async_api import async_playwright


async def extract_car_data(page, url):

    try:

        await page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=60000
        )

        await page.wait_for_timeout(1500)

        json_ld_scripts = await page.locator(
            'script[type="application/ld+json"]'
        ).all_text_contents()

        car = None

        for script in json_ld_scripts:

            try:

                data = json.loads(script)

                items = data if isinstance(data, list) else [data]

                for item in items:

                    if (
                        isinstance(item, dict)
                        and item.get("@type") == "Car"
                    ):
                        car = item
                        break

            except Exception:
                continue

            if car:
                break

        if not car:
            print("❌ No Car JSON-LD:", url)
            return None

        brand = car.get("brand", {})

        if isinstance(brand, dict):
            brand = brand.get("name")

        offer = car.get("offers", {})

        mileage = car.get(
            "mileageFromOdometer",
            {}
        )

        return {
            "listing_url": url,
            "car_name": car.get("name"),
            "brand": brand,
            "model": car.get("model"),
            "manufacturing_year": car.get(
                "vehicleModelDate"
            ),
            "body_type": car.get("bodyType"),
            "fuel_type": car.get("fuelType"),
            "transmission": car.get(
                "vehicleTransmission"
            ),
            "price_numeric": offer.get("price"),
            "price_currency": offer.get(
                "priceCurrency"
            ),
            "km_driven": mileage.get("value"),
        }

    except Exception as e:

        print("❌ Error:", url)
        print(str(e))

        return None


async def scrape_test_listings(urls):

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=True
        )

        page = await browser.new_page(
            viewport={
                "width": 1440,
                "height": 900
            }
        )

        results = []

        for i, url in enumerate(urls, 1):

            print(
                f"Scraping {i}/{len(urls)}"
            )

            data = await extract_car_data(
                page,
                url
            )

            if data:
                results.append(data)

            # Small delay between pages
            await asyncio.sleep(1)

        await browser.close()

        return results


test_data = await scrape_test_listings(
    listing_urls
)

df_test_20 = pd.DataFrame(test_data)

print("\n==============================")
print("RESULT")
print("==============================")
print("URLs:", len(listing_urls))
print("Successfully extracted:", len(df_test_20))

display(df_test_20)

Scraping 1/20
Scraping 2/20
Scraping 3/20
Scraping 4/20
Scraping 5/20
Scraping 6/20
Scraping 7/20
Scraping 8/20
Scraping 9/20
Scraping 10/20
Scraping 11/20
Scraping 12/20
Scraping 13/20
Scraping 14/20
Scraping 15/20
Scraping 16/20
Scraping 17/20
Scraping 18/20
Scraping 19/20
Scraping 20/20

RESULT
URLs: 20
Successfully extracted: 20


,listing_url,car_name,brand,model,manufacturing_year,body_type,fuel_type,transmission,price_numeric,price_currency,km_driven
0,https://www.cars24.com/buy-used-maruti-swift-2...,2024 Maruti Swift VXi (O),Maruti,Swift,2024,Hatchback,Petrol,Manual,695840,INR,15400
1,https://www.cars24.com/buy-used-renault-kwid-2...,2017 Renault Kwid CLIMBER 1.0,Renault,Kwid,2017,Hatchback,Petrol,Manual,270614,INR,71447
2,https://www.cars24.com/buy-used-maruti-swift-2...,2018 Maruti Swift VDI,Maruti,Swift,2018,Hatchback,Diesel,Manual,562075,INR,83771
3,https://www.cars24.com/buy-used-tata-nexon-202...,2022 Tata NEXON XM SUNROOF PETROL,Tata,NEXON,2022,SUV,Petrol,Manual,654840,INR,54649
4,https://www.cars24.com/buy-used-renault-kwid-2...,2020 Renault Kwid CLIMBER 1.0 (O),Renault,Kwid,2020,Hatchback,Petrol,Manual,369940,INR,33425
5,https://www.cars24.com/buy-used-tata-nexon-202...,2020 Tata NEXON XZ PLUS (O) PETROL,Tata,NEXON,2020,SUV,Petrol,Manual,716403,INR,89034
6,https://www.cars24.com/buy-used-volkswagen-pol...,2017 Volkswagen Polo HIGHLINE PLUS 1.2(16 ALLOY,Volkswagen,Polo,2017,Hatchback,Petrol,Manual,515246,INR,41547
7,https://www.cars24.com/buy-used-hyundai-elite-...,2017 Hyundai Elite i20 SPORTZ 1.2,Hyundai,Elite i20,2017,Hatchback,Petrol,Manual,467376,INR,57809
8,https://www.cars24.com/buy-used-ford-ecosport-...,2019 Ford Ecosport TITANIUM 1.5L PETROL,Ford,Ecosport,2019,SUV,Petrol,Manual,515576,INR,78135
9,https://www.cars24.com/buy-used-tata-altroz-20...,2021 Tata ALTROZ XT I-TURBO PETROL,Tata,ALTROZ,2021,Hatchback,Petrol,Manual,613840,INR,104414


In [11]:
import re
from playwright.async_api import async_playwright

url = "https://www.cars24.com/buy-used-cars-bangalore/"

async with async_playwright() as p:

    browser = await p.chromium.launch(headless=True)

    page = await browser.new_page(
        viewport={"width": 1440, "height": 900}
    )

    print("Opening Bangalore...")

    await page.goto(
        url,
        wait_until="domcontentloaded",
        timeout=60000
    )

    await page.wait_for_timeout(5000)

    for i in range(10):
        await page.mouse.wheel(0, 4000)
        await page.wait_for_timeout(2000)
        print(f"Scroll {i+1}/10")

    links = await page.locator(
        "a[href]"
    ).evaluate_all(
        """els => els.map(a => a.href)"""
    )

    listing_urls = []

    for href in links:

        href = href.split("?")[0].rstrip("/")

        if re.search(r"-\d{8,}$", href):
            listing_urls.append(href + "/")

    listing_urls = list(dict.fromkeys(listing_urls))

    print("\n==============================")
    print("TOTAL LINKS:", len(links))
    print("INDIVIDUAL LISTINGS:", len(listing_urls))
    print("==============================")

    for i, listing in enumerate(listing_urls[:20], 1):
        print(i, listing)

    await browser.close()

Opening Bangalore...
Scroll 1/10
Scroll 2/10
Scroll 3/10
Scroll 4/10
Scroll 5/10
Scroll 6/10
Scroll 7/10
Scroll 8/10
Scroll 9/10
Scroll 10/10

TOTAL LINKS: 502
INDIVIDUAL LISTINGS: 20
1 https://www.cars24.com/buy-used-renault-kwid-2025-cars-bangalore-13189872131/
2 https://www.cars24.com/buy-used-maruti-swift-2026-cars-bangalore-10285571135/
3 https://www.cars24.com/buy-used-maruti-vitara-brezza-2022-cars-bangalore-10036890169/
4 https://www.cars24.com/buy-used-hyundai-creta-2017-cars-bangalore-10212775137/
5 https://www.cars24.com/buy-used-renault-kwid-2019-cars-bangalore-11124674158/
6 https://www.cars24.com/buy-used-maruti-swift-2021-cars-bangalore-10213479135/
7 https://www.cars24.com/buy-used-maruti-vitara-brezza-2022-cars-bangalore-10079972150/
8 https://www.cars24.com/buy-used-maruti-ignis-2021-cars-bangalore-11128274157/
9 https://www.cars24.com/buy-used-tata-nexon-2021-cars-bangalore-10202069755/
10 https://www.cars24.com/buy-used-tata-nexon-2018-cars-bangalore-23584671192/
11

In [12]:
# Save Bangalore URLs
bangalore_urls = listing_urls if "listing_urls" in globals() else []

print("Bangalore URLs saved:", len(bangalore_urls))

for i, url in enumerate(bangalore_urls, 1):
    print(i, url)

Bangalore URLs saved: 20
1 https://www.cars24.com/buy-used-renault-kwid-2025-cars-bangalore-13189872131/
2 https://www.cars24.com/buy-used-maruti-swift-2026-cars-bangalore-10285571135/
3 https://www.cars24.com/buy-used-maruti-vitara-brezza-2022-cars-bangalore-10036890169/
4 https://www.cars24.com/buy-used-hyundai-creta-2017-cars-bangalore-10212775137/
5 https://www.cars24.com/buy-used-renault-kwid-2019-cars-bangalore-11124674158/
6 https://www.cars24.com/buy-used-maruti-swift-2021-cars-bangalore-10213479135/
7 https://www.cars24.com/buy-used-maruti-vitara-brezza-2022-cars-bangalore-10079972150/
8 https://www.cars24.com/buy-used-maruti-ignis-2021-cars-bangalore-11128274157/
9 https://www.cars24.com/buy-used-tata-nexon-2021-cars-bangalore-10202069755/
10 https://www.cars24.com/buy-used-tata-nexon-2018-cars-bangalore-23584671192/
11 https://www.cars24.com/buy-used-ford-ecosport-2019-cars-bangalore-10256378133/
12 https://www.cars24.com/buy-used-renault-kwid-2020-cars-bangalore-10042777191

In [13]:
# Check Bangalore URLs

print("Total Bangalore URLs:", len(bangalore_urls))

# Remove duplicates
bangalore_urls = list(dict.fromkeys(bangalore_urls))

print("Unique Bangalore URLs:", len(bangalore_urls))

# Show all URLs
for i, url in enumerate(bangalore_urls, 1):
    print(i, url)

Total Bangalore URLs: 20
Unique Bangalore URLs: 20
1 https://www.cars24.com/buy-used-renault-kwid-2025-cars-bangalore-13189872131/
2 https://www.cars24.com/buy-used-maruti-swift-2026-cars-bangalore-10285571135/
3 https://www.cars24.com/buy-used-maruti-vitara-brezza-2022-cars-bangalore-10036890169/
4 https://www.cars24.com/buy-used-hyundai-creta-2017-cars-bangalore-10212775137/
5 https://www.cars24.com/buy-used-renault-kwid-2019-cars-bangalore-11124674158/
6 https://www.cars24.com/buy-used-maruti-swift-2021-cars-bangalore-10213479135/
7 https://www.cars24.com/buy-used-maruti-vitara-brezza-2022-cars-bangalore-10079972150/
8 https://www.cars24.com/buy-used-maruti-ignis-2021-cars-bangalore-11128274157/
9 https://www.cars24.com/buy-used-tata-nexon-2021-cars-bangalore-10202069755/
10 https://www.cars24.com/buy-used-tata-nexon-2018-cars-bangalore-23584671192/
11 https://www.cars24.com/buy-used-ford-ecosport-2019-cars-bangalore-10256378133/
12 https://www.cars24.com/buy-used-renault-kwid-2020-

In [14]:
# Save the current Bangalore URLs separately
bangalore_listing_urls = bangalore_urls.copy()

print("Bangalore listings saved:", len(bangalore_listing_urls))

Bangalore listings saved: 20


In [15]:
import re
from playwright.async_api import async_playwright

hyderabad_url = "https://www.cars24.com/buy-used-cars-hyderabad/"

async with async_playwright() as p:

    browser = await p.chromium.launch(headless=True)

    page = await browser.new_page(
        viewport={"width": 1440, "height": 900}
    )

    print("Opening Hyderabad...")

    await page.goto(
        hyderabad_url,
        wait_until="domcontentloaded",
        timeout=60000
    )

    await page.wait_for_timeout(5000)

    for i in range(10):
        await page.mouse.wheel(0, 4000)
        await page.wait_for_timeout(2000)
        print(f"Scroll {i + 1}/10")

    links = await page.locator(
        "a[href]"
    ).evaluate_all(
        """els => els.map(a => a.href)"""
    )

    hyderabad_listing_urls = []

    for href in links:

        href = href.split("?")[0].rstrip("/")

        if re.search(r"-\d{8,}$", href):
            hyderabad_listing_urls.append(
                href + "/"
            )

    hyderabad_listing_urls = list(
        dict.fromkeys(hyderabad_listing_urls)
    )

    print("\n==============================")
    print(
        "Valid Hyderabad listings:",
        len(hyderabad_listing_urls)
    )
    print("==============================")

    for i, url in enumerate(
        hyderabad_listing_urls, 1
    ):
        print(i, url)

    await browser.close()

Opening Hyderabad...
Scroll 1/10
Scroll 2/10
Scroll 3/10
Scroll 4/10
Scroll 5/10
Scroll 6/10
Scroll 7/10
Scroll 8/10
Scroll 9/10
Scroll 10/10

Valid Hyderabad listings: 20
1 https://www.cars24.com/buy-used-maruti-swift-2024-cars-hyderabad-10440373131/
2 https://www.cars24.com/buy-used-renault-kwid-2017-cars-hyderabad-10473477137/
3 https://www.cars24.com/buy-used-maruti-swift-2018-cars-hyderabad-10432970186/
4 https://www.cars24.com/buy-used-tata-nexon-2022-cars-hyderabad-10425496120/
5 https://www.cars24.com/buy-used-renault-kwid-2020-cars-hyderabad-10457877130/
6 https://www.cars24.com/buy-used-tata-nexon-2020-cars-hyderabad-10420372181/
7 https://www.cars24.com/buy-used-volkswagen-polo-2017-cars-hyderabad-10420673116/
8 https://www.cars24.com/buy-used-hyundai-elite-i20-2017-cars-hyderabad-10488773188/
9 https://www.cars24.com/buy-used-ford-ecosport-2019-cars-hyderabad-10481879115/
10 https://www.cars24.com/buy-used-tata-altroz-2021-cars-hyderabad-12218574193/
11 https://www.cars24.c

In [16]:
# Combine Hyderabad + Bangalore URLs

all_listing_urls = (
    hyderabad_listing_urls
    + bangalore_listing_urls
)

# Remove duplicate URLs
all_listing_urls = list(
    dict.fromkeys(all_listing_urls)
)

print("Hyderabad listings:", len(hyderabad_listing_urls))
print("Bangalore listings:", len(bangalore_listing_urls))
print("Total unique listings:", len(all_listing_urls))

print("\nFirst 10 URLs:")
for i, url in enumerate(all_listing_urls[:10], 1):
    print(i, url)

Hyderabad listings: 20
Bangalore listings: 20
Total unique listings: 40

First 10 URLs:
1 https://www.cars24.com/buy-used-maruti-swift-2024-cars-hyderabad-10440373131/
2 https://www.cars24.com/buy-used-renault-kwid-2017-cars-hyderabad-10473477137/
3 https://www.cars24.com/buy-used-maruti-swift-2018-cars-hyderabad-10432970186/
4 https://www.cars24.com/buy-used-tata-nexon-2022-cars-hyderabad-10425496120/
5 https://www.cars24.com/buy-used-renault-kwid-2020-cars-hyderabad-10457877130/
6 https://www.cars24.com/buy-used-tata-nexon-2020-cars-hyderabad-10420372181/
7 https://www.cars24.com/buy-used-volkswagen-polo-2017-cars-hyderabad-10420673116/
8 https://www.cars24.com/buy-used-hyundai-elite-i20-2017-cars-hyderabad-10488773188/
9 https://www.cars24.com/buy-used-ford-ecosport-2019-cars-hyderabad-10481879115/
10 https://www.cars24.com/buy-used-tata-altroz-2021-cars-hyderabad-12218574193/


In [17]:
import asyncio
import json
import pandas as pd
from playwright.async_api import async_playwright


async def scrape_car(url, page):

    try:
        await page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=60000
        )

        await page.wait_for_timeout(1500)

        scripts = await page.locator(
            'script[type="application/ld+json"]'
        ).all_text_contents()

        car = None

        for script in scripts:

            try:
                data = json.loads(script)

                items = data if isinstance(data, list) else [data]

                for item in items:

                    if (
                        isinstance(item, dict)
                        and item.get("@type") == "Car"
                    ):
                        car = item
                        break

            except Exception:
                continue

            if car:
                break

        if not car:
            print("❌ No car data:", url)
            return None

        brand = car.get("brand", {})

        if isinstance(brand, dict):
            brand = brand.get("name")

        offer = car.get("offers", {})
        mileage = car.get("mileageFromOdometer", {})

        return {
            "listing_url": url,
            "car_name": car.get("name"),
            "brand": brand,
            "model": car.get("model"),
            "manufacturing_year": car.get("vehicleModelDate"),
            "body_type": car.get("bodyType"),
            "fuel_type": car.get("fuelType"),
            "transmission": car.get("vehicleTransmission"),
            "price_numeric": offer.get("price"),
            "price_currency": offer.get("priceCurrency"),
            "km_driven": mileage.get("value")
        }

    except Exception as e:
        print("❌ Error:", url)
        print(e)
        return None


async def scrape_all_40(urls):

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=True
        )

        page = await browser.new_page(
            viewport={
                "width": 1440,
                "height": 900
            }
        )

        results = []

        for i, url in enumerate(urls, 1):

            print(
                f"Scraping {i}/{len(urls)}"
            )

            data = await scrape_car(
                url,
                page
            )

            if data:
                results.append(data)

            await asyncio.sleep(1)

        await browser.close()

        return results


cars_40 = await scrape_all_40(
    all_listing_urls
)

df_40 = pd.DataFrame(cars_40)

print("\n==============================")
print("SCRAPING COMPLETE")
print("==============================")
print("URLs:", len(all_listing_urls))
print("Records:", len(df_40))
print("Duplicate URLs:", df_40["listing_url"].duplicated().sum())

display(df_40)

Scraping 1/40
Scraping 2/40
Scraping 3/40
Scraping 4/40
Scraping 5/40
Scraping 6/40
Scraping 7/40
Scraping 8/40
Scraping 9/40
Scraping 10/40
Scraping 11/40
Scraping 12/40
Scraping 13/40
Scraping 14/40
Scraping 15/40
Scraping 16/40
Scraping 17/40
Scraping 18/40
Scraping 19/40
Scraping 20/40
Scraping 21/40
Scraping 22/40
Scraping 23/40
Scraping 24/40
Scraping 25/40
Scraping 26/40
Scraping 27/40
Scraping 28/40
Scraping 29/40
Scraping 30/40
Scraping 31/40
Scraping 32/40
Scraping 33/40
Scraping 34/40
Scraping 35/40
Scraping 36/40
Scraping 37/40
Scraping 38/40
Scraping 39/40
Scraping 40/40

SCRAPING COMPLETE
URLs: 40
Records: 40
Duplicate URLs: 0


,listing_url,car_name,brand,model,manufacturing_year,body_type,fuel_type,transmission,price_numeric,price_currency,km_driven
0,https://www.cars24.com/buy-used-maruti-swift-2...,2024 Maruti Swift VXi (O),Maruti,Swift,2024,Hatchback,Petrol,Manual,695840,INR,15400
1,https://www.cars24.com/buy-used-renault-kwid-2...,2017 Renault Kwid CLIMBER 1.0,Renault,Kwid,2017,Hatchback,Petrol,Manual,270614,INR,71447
2,https://www.cars24.com/buy-used-maruti-swift-2...,2018 Maruti Swift VDI,Maruti,Swift,2018,Hatchback,Diesel,Manual,562075,INR,83771
3,https://www.cars24.com/buy-used-tata-nexon-202...,2022 Tata NEXON XM SUNROOF PETROL,Tata,NEXON,2022,SUV,Petrol,Manual,654840,INR,54649
4,https://www.cars24.com/buy-used-renault-kwid-2...,2020 Renault Kwid CLIMBER 1.0 (O),Renault,Kwid,2020,Hatchback,Petrol,Manual,369940,INR,33425
5,https://www.cars24.com/buy-used-tata-nexon-202...,2020 Tata NEXON XZ PLUS (O) PETROL,Tata,NEXON,2020,SUV,Petrol,Manual,716403,INR,89034
6,https://www.cars24.com/buy-used-volkswagen-pol...,2017 Volkswagen Polo HIGHLINE PLUS 1.2(16 ALLOY,Volkswagen,Polo,2017,Hatchback,Petrol,Manual,515246,INR,41547
7,https://www.cars24.com/buy-used-hyundai-elite-...,2017 Hyundai Elite i20 SPORTZ 1.2,Hyundai,Elite i20,2017,Hatchback,Petrol,Manual,467376,INR,57809
8,https://www.cars24.com/buy-used-ford-ecosport-...,2019 Ford Ecosport TITANIUM 1.5L PETROL,Ford,Ecosport,2019,SUV,Petrol,Manual,515576,INR,78135
9,https://www.cars24.com/buy-used-tata-altroz-20...,2021 Tata ALTROZ XT I-TURBO PETROL,Tata,ALTROZ,2021,Hatchback,Petrol,Manual,613840,INR,104414


In [18]:
# Save our validated 40-record dataset

df_40.to_csv(
    "cars24_validated_40_raw.csv",
    index=False
)

print("Saved:", "cars24_validated_40_raw.csv")
print("Records:", len(df_40))
print("Columns:", len(df_40.columns))

Saved: cars24_validated_40_raw.csv
Records: 40
Columns: 11


In [19]:
import re
import asyncio
from playwright.async_api import async_playwright

CITY_URLS = {
    "Hyderabad": "https://www.cars24.com/buy-used-cars-hyderabad/",
    "Bangalore": "https://www.cars24.com/buy-used-cars-bangalore/",
    "Chennai": "https://www.cars24.com/buy-used-cars-chennai/",
    "Pune": "https://www.cars24.com/buy-used-cars-pune/",
    "Delhi": "https://www.cars24.com/buy-used-cars-delhi/",
    "Mumbai": "https://www.cars24.com/buy-used-cars-mumbai/",
    "Kolkata": "https://www.cars24.com/buy-used-cars-kolkata/",
    "Ahmedabad": "https://www.cars24.com/buy-used-cars-ahmedabad/",
    "Noida": "https://www.cars24.com/buy-used-cars-noida/",
    "Gurgaon": "https://www.cars24.com/buy-used-cars-gurgaon/"
}


def is_individual_listing(url):

    url = url.split("?")[0].rstrip("/")

    return bool(
        re.search(r"-\d{8,}$", url)
    )


async def collect_city_urls(page, city, city_url):

    print(f"\n========== {city} ==========")

    await page.goto(
        city_url,
        wait_until="domcontentloaded",
        timeout=60000
    )

    await page.wait_for_timeout(5000)

    # Scroll several times to load rendered content
    for i in range(10):

        await page.mouse.wheel(0, 4000)

        await page.wait_for_timeout(1500)

    links = await page.locator(
        "a[href]"
    ).evaluate_all(
        """els => els.map(a => a.href)"""
    )

    valid_urls = []

    for href in links:

        href = href.split("?")[0].rstrip("/")

        if is_individual_listing(href):

            valid_urls.append(
                href + "/"
            )

    valid_urls = list(
        dict.fromkeys(valid_urls)
    )

    print("Links found:", len(links))
    print("Valid listing URLs:", len(valid_urls))

    return valid_urls


async def collect_all_cities():

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=True
        )

        page = await browser.new_page(
            viewport={
                "width": 1440,
                "height": 900
            }
        )

        all_urls = {}

        for city, city_url in CITY_URLS.items():

            try:

                urls = await collect_city_urls(
                    page,
                    city,
                    city_url
                )

                all_urls[city] = urls

                await asyncio.sleep(2)

            except Exception as e:

                print(
                    f"ERROR in {city}: {e}"
                )

                all_urls[city] = []

        await browser.close()

        return all_urls


city_listing_urls = await collect_all_cities()


print("\n\n==============================")
print("FINAL URL COLLECTION")
print("==============================")

total = 0

for city, urls in city_listing_urls.items():

    print(
        f"{city}: {len(urls)} listings"
    )

    total += len(urls)

print("------------------------------")
print("TOTAL BEFORE DEDUP:", total)


# Combine all cities
all_listing_urls_2k = []

for urls in city_listing_urls.values():

    all_listing_urls_2k.extend(urls)


# Global deduplication
all_listing_urls_2k = list(
    dict.fromkeys(all_listing_urls_2k)
)

print(
    "TOTAL UNIQUE LISTING URLs:",
    len(all_listing_urls_2k)
)



========== Hyderabad ==========
Links found: 499
Valid listing URLs: 20

========== Bangalore ==========
Links found: 502
Valid listing URLs: 20

========== Chennai ==========
Links found: 464
Valid listing URLs: 20

========== Pune ==========
Links found: 489
Valid listing URLs: 20

========== Delhi ==========
Links found: 193
Valid listing URLs: 0

========== Mumbai ==========
Links found: 484
Valid listing URLs: 20

========== Kolkata ==========
Links found: 435
Valid listing URLs: 20

========== Ahmedabad ==========
Links found: 462
Valid listing URLs: 20

========== Noida ==========
Links found: 459
Valid listing URLs: 40

========== Gurgaon ==========
Links found: 477
Valid listing URLs: 40


FINAL URL COLLECTION
Hyderabad: 20 listings
Bangalore: 20 listings
Chennai: 20 listings
Pune: 20 listings
Delhi: 0 listings
Mumbai: 20 listings
Kolkata: 20 listings
Ahmedabad: 20 listings
Noida: 40 listings
Gurgaon: 40 listings
------------------------------
TOTAL BEFORE DEDUP: 220
TOTAL UN

In [20]:
# Save collected listing URLs

with open("cars24_listing_urls_194.txt", "w") as f:
    for url in all_listing_urls_2k:
        f.write(url + "\n")

print("Saved URLs:", len(all_listing_urls_2k))
print("File: cars24_listing_urls_194.txt")

Saved URLs: 194
File: cars24_listing_urls_194.txt


In [21]:
import re
import asyncio
from playwright.async_api import async_playwright

CATEGORY_URLS = {
    "Maruti": "https://www.cars24.com/buy-used-maruti-cars/",
    "Hyundai": "https://www.cars24.com/buy-used-hyundai-cars/",
    "Tata": "https://www.cars24.com/buy-used-tata-cars/",
    "Honda": "https://www.cars24.com/buy-used-honda-cars/",
    "Mahindra": "https://www.cars24.com/buy-used-mahindra-cars/",
    "Renault": "https://www.cars24.com/buy-used-renault-cars/",
    "Kia": "https://www.cars24.com/buy-used-kia-cars/",
    "Ford": "https://www.cars24.com/buy-used-ford-cars/",
    "Toyota": "https://www.cars24.com/buy-used-toyota-cars/",
    "Volkswagen": "https://www.cars24.com/buy-used-volkswagen-cars/"
}


def is_listing_url(url):

    url = url.split("?")[0].rstrip("/")

    return bool(
        re.search(r"-\d{8,}$", url)
    )


async def collect_category_urls(page, brand, category_url):

    print(f"\n========== {brand} ==========")

    await page.goto(
        category_url,
        wait_until="domcontentloaded",
        timeout=60000
    )

    await page.wait_for_timeout(5000)

    for i in range(10):

        await page.mouse.wheel(0, 4000)

        await page.wait_for_timeout(1500)

    links = await page.locator(
        "a[href]"
    ).evaluate_all(
        """els => els.map(a => a.href)"""
    )

    urls = []

    for href in links:

        href = href.split("?")[0].rstrip("/")

        if is_listing_url(href):

            urls.append(href + "/")

    urls = list(dict.fromkeys(urls))

    print("Links found:", len(links))
    print("Listing URLs:", len(urls))

    return urls


async def collect_categories():

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=True
        )

        page = await browser.new_page(
            viewport={
                "width": 1440,
                "height": 900
            }
        )

        results = {}

        for brand, url in CATEGORY_URLS.items():

            try:

                results[brand] = await collect_category_urls(
                    page,
                    brand,
                    url
                )

                await asyncio.sleep(2)

            except Exception as e:

                print(
                    f"ERROR in {brand}: {e}"
                )

                results[brand] = []

        await browser.close()

        return results


category_listing_urls = await collect_categories()


# Combine with our existing 194 URLs
expanded_urls = all_listing_urls_2k.copy()

for urls in category_listing_urls.values():
    expanded_urls.extend(urls)


# Remove duplicates
expanded_urls = list(
    dict.fromkeys(expanded_urls)
)


print("\n==============================")
print("EXPANDED COLLECTION")
print("==============================")

print(
    "Previous unique URLs:",
    len(all_listing_urls_2k)
)

print(
    "New unique URLs:",
    len(expanded_urls)
)

print(
    "New URLs discovered:",
    len(expanded_urls) - len(all_listing_urls_2k)
)


========== Maruti ==========
Links found: 444
Listing URLs: 20

========== Hyundai ==========
Links found: 444
Listing URLs: 20

========== Tata ==========
Links found: 444
Listing URLs: 20

========== Honda ==========
Links found: 444
Listing URLs: 20

========== Mahindra ==========
Links found: 443
Listing URLs: 20

========== Renault ==========
Links found: 443
Listing URLs: 20

========== Kia ==========
Links found: 445
Listing URLs: 20

========== Ford ==========
Links found: 443
Listing URLs: 20

========== Toyota ==========
Links found: 440
Listing URLs: 20

========== Volkswagen ==========
Links found: 443
Listing URLs: 20

EXPANDED COLLECTION
Previous unique URLs: 194
New unique URLs: 310
New URLs discovered: 116


In [22]:
from playwright.async_api import async_playwright
import re

BASE_URL = "https://www.cars24.com"

# Start with the URLs we already collected
all_urls = set(expanded_urls)

async def collect_from_page(page, url):
    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=60000)
        await page.wait_for_timeout(3000)

        # Scroll more times to load available listings
        for _ in range(15):
            await page.mouse.wheel(0, 2500)
            await page.wait_for_timeout(1000)

        hrefs = await page.locator("a[href]").evaluate_all(
            """els => els.map(e => e.href).filter(Boolean)"""
        )

        listing_urls = set()

        for href in hrefs:
            clean = href.split("?")[0].rstrip("/")

            # Cars24 individual listing URL heuristic
            if re.search(r"-\d{8,}$", clean):
                listing_urls.add(clean + "/")

        return listing_urls

    except Exception as e:
        print("ERROR:", url, "|", e)
        return set()


async def main():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        # Known Cars24 category/filter pages
        pages_to_check = [
            "https://www.cars24.com/buy-used-cars-hyderabad/",
            "https://www.cars24.com/buy-used-cars-bangalore/",
            "https://www.cars24.com/buy-used-cars-chennai/",
            "https://www.cars24.com/buy-used-cars-pune/",
            "https://www.cars24.com/buy-used-cars-mumbai/",
            "https://www.cars24.com/buy-used-cars-kolkata/",
            "https://www.cars24.com/buy-used-cars-ahmedabad/",
            "https://www.cars24.com/buy-used-cars-noida/",
            "https://www.cars24.com/buy-used-cars-gurgaon/",

            "https://www.cars24.com/buy-used-maruti-cars/",
            "https://www.cars24.com/buy-used-hyundai-cars/",
            "https://www.cars24.com/buy-used-tata-cars/",
            "https://www.cars24.com/buy-used-honda-cars/",
            "https://www.cars24.com/buy-used-mahindra-cars/",
            "https://www.cars24.com/buy-used-renault-cars/",
            "https://www.cars24.com/buy-used-kia-cars/",
            "https://www.cars24.com/buy-used-ford-cars/",
            "https://www.cars24.com/buy-used-toyota-cars/",
            "https://www.cars24.com/buy-used-volkswagen-cars/",
        ]

        before = len(all_urls)

        for url in pages_to_check:
            print("\nChecking:", url)

            found = await collect_from_page(page, url)

            new_urls = found - all_urls
            all_urls.update(found)

            print("Found:", len(found))
            print("New:", len(new_urls))
            print("Total unique:", len(all_urls))

            if len(all_urls) >= 2000:
                print("\n🎯 TARGET REACHED:", len(all_urls))
                break

        await browser.close()

    print("\n==============================")
    print("FINAL URL COLLECTION")
    print("==============================")
    print("Previous:", before)
    print("New:", len(all_urls) - before)
    print("Total unique:", len(all_urls))

    with open("cars24_listing_urls_expanded.txt", "w") as f:
        for url in sorted(all_urls):
            f.write(url + "\n")

    print("Saved: cars24_listing_urls_expanded.txt")


await main()

# Keep variable available for next cells
expanded_urls = all_urls


Checking: https://www.cars24.com/buy-used-cars-hyderabad/
Found: 100
New: 80
Total unique: 390

Checking: https://www.cars24.com/buy-used-cars-bangalore/
Found: 100
New: 70
Total unique: 460

Checking: https://www.cars24.com/buy-used-cars-chennai/
Found: 100
New: 80
Total unique: 540

Checking: https://www.cars24.com/buy-used-cars-pune/
Found: 100
New: 78
Total unique: 618

Checking: https://www.cars24.com/buy-used-cars-mumbai/
Found: 100
New: 80
Total unique: 698

Checking: https://www.cars24.com/buy-used-cars-kolkata/
Found: 100
New: 77
Total unique: 775

Checking: https://www.cars24.com/buy-used-cars-ahmedabad/
Found: 100
New: 73
Total unique: 848

Checking: https://www.cars24.com/buy-used-cars-noida/
Found: 160
New: 101
Total unique: 949

Checking: https://www.cars24.com/buy-used-cars-gurgaon/
Found: 160
New: 80
Total unique: 1029

Checking: https://www.cars24.com/buy-used-maruti-cars/
Found: 40
New: 7
Total unique: 1036

Checking: https://www.cars24.com/buy-used-hyundai-cars/
Fou

In [23]:
MORE_CITY_URLS = {
    "Jaipur": "https://www.cars24.com/buy-used-cars-jaipur/",
    "Lucknow": "https://www.cars24.com/buy-used-cars-lucknow/",
    "Chandigarh": "https://www.cars24.com/buy-used-cars-chandigarh/",
    "Indore": "https://www.cars24.com/buy-used-cars-indore/",
    "Surat": "https://www.cars24.com/buy-used-cars-surat/",
    "Kochi": "https://www.cars24.com/buy-used-cars-kochi/",
    "Coimbatore": "https://www.cars24.com/buy-used-cars-coimbatore/",
    "Nagpur": "https://www.cars24.com/buy-used-cars-nagpur/",
    "Patna": "https://www.cars24.com/buy-used-cars-patna/",
    "Bhopal": "https://www.cars24.com/buy-used-cars-bhopal/",
    "Vadodara": "https://www.cars24.com/buy-used-cars-vadodara/",
    "Nashik": "https://www.cars24.com/buy-used-cars-nashik/",
}

async def collect_more_cities():

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        before = len(expanded_urls)

        for city, url in MORE_CITY_URLS.items():

            print(f"\n========== {city} ==========")

            found = await collect_from_page(page, url)

            new_urls = found - expanded_urls

            expanded_urls.update(found)

            print("Found:", len(found))
            print("New:", len(new_urls))
            print("Total unique:", len(expanded_urls))

            if len(expanded_urls) >= 2000:
                print("\n🎯 2000+ TARGET REACHED!")
                break

        await browser.close()

    print("\n==============================")
    print("UPDATED URL COLLECTION")
    print("==============================")
    print("Previous:", before)
    print("New:", len(expanded_urls) - before)
    print("Total unique:", len(expanded_urls))

    with open("cars24_listing_urls_2000_target.txt", "w") as f:
        for url in sorted(expanded_urls):
            f.write(url + "\n")

    print("Saved: cars24_listing_urls_2000_target.txt")


await collect_more_cities()


========== Jaipur ==========
Found: 100
New: 85
Total unique: 1217

========== Lucknow ==========
Found: 100
New: 80
Total unique: 1297

========== Chandigarh ==========
Found: 147
New: 133
Total unique: 1430

========== Indore ==========
Found: 100
New: 83
Total unique: 1513

========== Surat ==========
Found: 100
New: 90
Total unique: 1603

========== Kochi ==========
Found: 100
New: 81
Total unique: 1684

========== Coimbatore ==========
Found: 100
New: 83
Total unique: 1767

========== Nagpur ==========
Found: 40
New: 40
Total unique: 1807

========== Patna ==========
Found: 100
New: 83
Total unique: 1890

========== Bhopal ==========
Found: 100
New: 91
Total unique: 1981

========== Vadodara ==========
Found: 100
New: 89
Total unique: 2070

🎯 2000+ TARGET REACHED!

UPDATED URL COLLECTION
Previous: 1132
New: 938
Total unique: 2070
Saved: cars24_listing_urls_2000_target.txt


In [24]:
import json
import pandas as pd
from datetime import datetime
from playwright.async_api import async_playwright

# Load collected URLs
with open("cars24_listing_urls_2000_target.txt", "r") as f:
    listing_urls = [line.strip() for line in f if line.strip()]

print("Total URLs loaded:", len(listing_urls))


def find_car_jsonld(json_data):
    """Find Car object inside JSON-LD."""
    if isinstance(json_data, dict):
        if json_data.get("@type") == "Car":
            return json_data

        for value in json_data.values():
            result = find_car_jsonld(value)
            if result:
                return result

    elif isinstance(json_data, list):
        for item in json_data:
            result = find_car_jsonld(item)
            if result:
                return result

    return None


def get_nested(data, *keys):
    """Safely read nested JSON fields."""
    current = data

    for key in keys:
        if not isinstance(current, dict):
            return None
        current = current.get(key)

    return current


async def extract_car(page, url):

    try:
        await page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=60000
        )

        await page.wait_for_timeout(1500)

        scripts = page.locator('script[type="application/ld+json"]')

        count = await scripts.count()

        car = None

        for i in range(count):

            text = await scripts.nth(i).text_content()

            if not text:
                continue

            try:
                data = json.loads(text)
                car = find_car_jsonld(data)

                if car:
                    break

            except:
                continue

        if not car:
            return None

        offers = car.get("offers", {})

        mileage = car.get("mileageFromOdometer", {})

        brand = car.get("brand")

        if isinstance(brand, dict):
            brand = brand.get("name")

        record = {

            "listing_url": url,

            "car_name": car.get("name"),

            "brand": brand,

            "model": car.get("model"),

            "manufacturing_year":
                car.get("vehicleModelDate"),

            "body_type":
                car.get("bodyType"),

            "fuel_type":
                car.get("fuelType"),

            "transmission":
                car.get("vehicleTransmission"),

            "price_numeric":
                offers.get("price")
                if isinstance(offers, dict)
                else None,

            "price_currency":
                offers.get("priceCurrency")
                if isinstance(offers, dict)
                else None,

            "km_driven":
                mileage.get("value")
                if isinstance(mileage, dict)
                else None,

            "source_website": "Cars24",

            "scraped_date":
                datetime.now().strftime("%Y-%m-%d")
        }

        return record

    except Exception as e:

        print("FAILED:", url)
        print("Reason:", str(e)[:150])

        return None


async def test_scrape():

    records = []

    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=True)

        page = await browser.new_page()

        # ONLY FIRST 20 FOR TESTING
        test_urls = listing_urls[:20]

        for i, url in enumerate(test_urls, start=1):

            print(f"[{i}/{len(test_urls)}] Scraping...")

            record = await extract_car(page, url)

            if record:

                records.append(record)

                print(
                    "OK:",
                    record["car_name"],
                    "| ₹",
                    record["price_numeric"]
                )

            else:

                print("NO DATA")

        await browser.close()

    df_test = pd.DataFrame(records)

    df_test.to_csv(
        "cars24_20_scrape_test.csv",
        index=False
    )

    print("\n============================")
    print("TEST COMPLETE")
    print("============================")

    print("URLs tested:", len(test_urls))
    print("Successful:", len(df_test))
    print("Failed:", len(test_urls) - len(df_test))

    print("\nSaved: cars24_20_scrape_test.csv")

    return df_test


df_test = await test_scrape()

df_test

Total URLs loaded: 2070
[1/20] Scraping...
OK: 2021 Audi A4 40 TFSI TECHNOLOGY | ₹ 2524340
[2/20] Scraping...
OK: 2024 Audi A4 40 TFSI  PREMIUM PLUS | ₹ 3121340
[3/20] Scraping...
OK: 2014 Audi A6 2.0 TDI PREMIUM PLUS | ₹ 594500
[4/20] Scraping...
OK: 2019 Audi Q5 40 TDI PREMIUM PLUS | ₹ 2675340
[5/20] Scraping...
OK: 2014 BMW X1 SDRIVE 20D X LINE | ₹ 760000
[6/20] Scraping...
OK: 2020 BMW X1 SDrive20i Xline | ₹ 2096994
[7/20] Scraping...
OK: 2022 BMW X1 SDRIVE 20i SPORTX | ₹ 3349000
[8/20] Scraping...
OK: 2012 Chevrolet Beat LT PETROL | ₹ 90000
[9/20] Scraping...
OK: 2014 Chevrolet Beat LS PETROL | ₹ 133140
[10/20] Scraping...
OK: 2013 Chevrolet Cruze LTZ AT | ₹ 374597
[11/20] Scraping...
OK: 2013 Chevrolet Sail 1.2 LS ABS | ₹ 189140
[12/20] Scraping...
OK: 2013 Chevrolet Sail 1.2 LS ABS | ₹ 129936
[13/20] Scraping...
OK: 2014 Chevrolet Sail 1.2 LS | ₹ 161981
[14/20] Scraping...
OK: 2013 Chevrolet Sail UVA 1.2 BASE | ₹ 221936
[15/20] Scraping...
OK: 2022 CITROEN C3 FEEL 1.2 | ₹ 408227

,listing_url,car_name,brand,model,manufacturing_year,body_type,fuel_type,transmission,price_numeric,price_currency,km_driven,source_website,scraped_date
0,https://www.cars24.com/buy-used-audi-a4-2021-c...,2021 Audi A4 40 TFSI TECHNOLOGY,Audi,A4,2021,Sedan,Petrol,Automatic,2524340,INR,62836,Cars24,2026-09-21
1,https://www.cars24.com/buy-used-audi-a4-2024-c...,2024 Audi A4 40 TFSI PREMIUM PLUS,Audi,A4,2024,Sedan,Petrol,Automatic,3121340,INR,11198,Cars24,2026-09-21
2,https://www.cars24.com/buy-used-audi-a6-2014-c...,2014 Audi A6 2.0 TDI PREMIUM PLUS,Audi,A6,2014,Sedan,Diesel,Automatic,594500,INR,110048,Cars24,2026-09-21
3,https://www.cars24.com/buy-used-audi-q5-2019-c...,2019 Audi Q5 40 TDI PREMIUM PLUS,Audi,Q5,2019,SUV,Diesel,Automatic,2675340,INR,77788,Cars24,2026-09-21
4,https://www.cars24.com/buy-used-bmw-x1-2014-ca...,2014 BMW X1 SDRIVE 20D X LINE,BMW,X1,2014,SUV,Diesel,Automatic,760000,INR,99855,Cars24,2026-09-21
5,https://www.cars24.com/buy-used-bmw-x1-2020-ca...,2020 BMW X1 SDrive20i Xline,BMW,X1,2020,SUV,Petrol,Automatic,2096994,INR,58334,Cars24,2026-09-21
6,https://www.cars24.com/buy-used-bmw-x1-2022-ca...,2022 BMW X1 SDRIVE 20i SPORTX,BMW,X1,2022,SUV,Petrol,Automatic,3349000,INR,82679,Cars24,2026-09-21
7,https://www.cars24.com/buy-used-chevrolet-beat...,2012 Chevrolet Beat LT PETROL,Chevrolet,Beat,2012,Hatchback,Petrol,Manual,90000,INR,47972,Cars24,2026-09-21
8,https://www.cars24.com/buy-used-chevrolet-beat...,2014 Chevrolet Beat LS PETROL,Chevrolet,Beat,2014,Hatchback,Petrol,Manual,133140,INR,78601,Cars24,2026-09-21
9,https://www.cars24.com/buy-used-chevrolet-cruz...,2013 Chevrolet Cruze LTZ AT,Chevrolet,Cruze,2013,Sedan,Diesel,Automatic,374597,INR,111286,Cars24,2026-09-21


In [25]:
import os
import pandas as pd
from datetime import datetime
from playwright.async_api import async_playwright

# Load all URLs
with open("cars24_listing_urls_2000_target.txt", "r") as f:
    listing_urls = [line.strip() for line in f if line.strip()]

print("Total URLs:", len(listing_urls))

CHECKPOINT_FILE = "cars24_raw_checkpoint.csv"
FINAL_FILE = "cars24_raw_2070.csv"

# Resume if checkpoint already exists
if os.path.exists(CHECKPOINT_FILE):
    old_df = pd.read_csv(CHECKPOINT_FILE)
    scraped_urls = set(old_df["listing_url"].astype(str))

    records = old_df.to_dict("records")

    print("Existing checkpoint:", len(records))
    print("Remaining:", len(listing_urls) - len(scraped_urls))

else:
    records = []
    scraped_urls = set()

    print("Starting fresh scrape")


async def scrape_all():

    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=True)

        page = await browser.new_page()

        remaining_urls = [
            url for url in listing_urls
            if url not in scraped_urls
        ]

        total_remaining = len(remaining_urls)

        for index, url in enumerate(remaining_urls, start=1):

            try:

                record = await extract_car(page, url)

                if record:

                    records.append(record)
                    scraped_urls.add(url)

                    print(
                        f"[{index}/{total_remaining}] "
                        f"OK | {record['car_name']}"
                    )

                else:

                    print(
                        f"[{index}/{total_remaining}] "
                        f"NO DATA"
                    )

            except Exception as e:

                print(
                    f"[{index}/{total_remaining}] "
                    f"ERROR | {str(e)[:100]}"
                )

            # Save checkpoint every 50 records
            if len(records) % 50 == 0 and len(records) > 0:

                checkpoint_df = pd.DataFrame(records)

                checkpoint_df.to_csv(
                    CHECKPOINT_FILE,
                    index=False
                )

                print(
                    f"\n💾 CHECKPOINT SAVED: "
                    f"{len(records)} records\n"
                )

        await browser.close()


await scrape_all()


# Final dataframe
df_raw = pd.DataFrame(records)

# Remove duplicate listing URLs
df_raw = df_raw.drop_duplicates(
    subset=["listing_url"]
).reset_index(drop=True)

# Save final raw dataset
df_raw.to_csv(
    FINAL_FILE,
    index=False
)

print("\n================================")
print("FULL SCRAPE COMPLETE")
print("================================")

print("Total URLs:", len(listing_urls))
print("Raw records:", len(df_raw))
print("Unique records:", df_raw["listing_url"].nunique())
print("Columns:", len(df_raw.columns))

print("\nSaved:")
print(FINAL_FILE)

print("\nData preview:")
display(df_raw.head())

Total URLs: 2070
Starting fresh scrape
[1/2070] OK | 2021 Audi A4 40 TFSI TECHNOLOGY
[2/2070] OK | 2024 Audi A4 40 TFSI  PREMIUM PLUS
[3/2070] OK | 2014 Audi A6 2.0 TDI PREMIUM PLUS
[4/2070] OK | 2019 Audi Q5 40 TDI PREMIUM PLUS
[5/2070] OK | 2014 BMW X1 SDRIVE 20D X LINE
[6/2070] OK | 2020 BMW X1 SDrive20i Xline
[7/2070] OK | 2022 BMW X1 SDRIVE 20i SPORTX
[8/2070] OK | 2012 Chevrolet Beat LT PETROL
[9/2070] OK | 2014 Chevrolet Beat LS PETROL
[10/2070] OK | 2013 Chevrolet Cruze LTZ AT
[11/2070] OK | 2013 Chevrolet Sail 1.2 LS ABS
[12/2070] OK | 2013 Chevrolet Sail 1.2 LS ABS
[13/2070] OK | 2014 Chevrolet Sail 1.2 LS
[14/2070] OK | 2013 Chevrolet Sail UVA 1.2 BASE
[15/2070] OK | 2022 CITROEN C3 FEEL 1.2
[16/2070] OK | 2026 CITROEN C3 SHINE 1.2
[17/2070] OK | 2017 Datsun Go ANNIVERSARY EDITION
[18/2070] OK | 2017 Datsun Go T
[19/2070] OK | 2018 Datsun Go T
[20/2070] OK | 2019 Datsun Go A(O)
[21/2070] OK | 2015 Datsun Go Plus T
[22/2070] OK | 2015 Datsun Go Plus T
[23/2070] OK | 2018 Dats

,listing_url,car_name,brand,model,manufacturing_year,body_type,fuel_type,transmission,price_numeric,price_currency,km_driven,source_website,scraped_date
0,https://www.cars24.com/buy-used-audi-a4-2021-c...,2021 Audi A4 40 TFSI TECHNOLOGY,Audi,A4,2021,Sedan,Petrol,Automatic,2524340,INR,62836,Cars24,2026-09-21
1,https://www.cars24.com/buy-used-audi-a4-2024-c...,2024 Audi A4 40 TFSI PREMIUM PLUS,Audi,A4,2024,Sedan,Petrol,Automatic,3121340,INR,11198,Cars24,2026-09-21
2,https://www.cars24.com/buy-used-audi-a6-2014-c...,2014 Audi A6 2.0 TDI PREMIUM PLUS,Audi,A6,2014,Sedan,Diesel,Automatic,594500,INR,110048,Cars24,2026-09-21
3,https://www.cars24.com/buy-used-audi-q5-2019-c...,2019 Audi Q5 40 TDI PREMIUM PLUS,Audi,Q5,2019,SUV,Diesel,Automatic,2675340,INR,77788,Cars24,2026-09-21
4,https://www.cars24.com/buy-used-bmw-x1-2014-ca...,2014 BMW X1 SDRIVE 20D X LINE,BMW,X1,2014,SUV,Diesel,Automatic,760000,INR,99855,Cars24,2026-09-21
